# CPU baseline for the full FDTD + renderer pipeline

This notebook extends `cpu_benchmark.ipynb` from solver-only timing to a full software pipeline on the PYNQ-Z1 Cortex-A9:

1. run the 2D FDTD update,
2. build the render magnitude buffer used as the heightmap,
3. ray-march a 640x480 frame with 48 samples per pixel and shade it into an RGB framebuffer.

The FDTD kernel matches the existing CPU benchmark: Q3.13 fixed point, two-pass Ey/Ex then Bz update, six-cell PML, and `cb = -717`. The renderer is a CPU reference workload for the whole display pipeline rather than a bit-exact RTL simulator. It performs the same class of work the FPGA avoids on the PS: full-frame ray marching, height sampling, normal estimation, and RGB framebuffer generation.

Run all cells on the PYNQ. The final cell compares the measured CPU pipeline FPS against the FPGA HDMI limit of 60 frames/s and the measured/assumed FDTD hardware rate.

In [ ]:
c_src = r'''
#include <stdint.h>
#include <string.h>
#include <math.h>

#define GRID 128
#define WIDTH 640
#define HEIGHT 480
#define PML 6
#define CB (-717)
#define RENDER_STEPS 48

static int16_t ey[GRID][GRID], ex[GRID][GRID], bz[GRID][GRID];
static int32_t ca_ey[GRID][GRID], ca_ex[GRID][GRID], ca_bz[GRID][GRID];
static uint16_t mag[GRID][GRID];
static uint8_t frame[HEIGHT][WIDTH][3];
static int iter_count = 0;

static inline int16_t q313(int32_t p) { return (int16_t)((p + 4096) >> 13); }

static inline uint16_t abs16(int16_t v) {
    int32_t x = v;
    return (uint16_t)(x < 0 ? -x : x);
}

static inline uint16_t sat16_u32(uint32_t v) {
    return (uint16_t)(v > 65535u ? 65535u : v);
}

static inline uint8_t sat8_float(float v) {
    if (v <= 0.0f) return 0;
    if (v >= 255.0f) return 255;
    return (uint8_t)(v + 0.5f);
}

static int depth(int i) {
    int lo = PML - 1 - i;
    int hi = i - (GRID - PML);
    int d = (lo > 0 ? lo : 0) + (hi > 0 ? hi : 0);
    return d > PML - 1 ? PML - 1 : d;
}

void init(const int32_t *ramp) {
    for (int r = 0; r < GRID; r++)
        for (int c = 0; c < GRID; c++) {
            int dr = depth(r), dc = depth(c);
            ca_ey[r][c] = ramp[dr];
            ca_ex[r][c] = ramp[dc];
            ca_bz[r][c] = ramp[dr > dc ? dr : dc];
        }
}

void reset(void) {
    memset(ey, 0, sizeof ey);
    memset(ex, 0, sizeof ex);
    memset(bz, 0, sizeof bz);
    memset(mag, 0, sizeof mag);
    memset(frame, 0, sizeof frame);
    iter_count = 0;
}

void run_fdtd(int iters, int nthreads) {
    for (int n = 0; n < iters; n++) {
        #pragma omp parallel for num_threads(nthreads) schedule(static)
        for (int r = 0; r < GRID; r++) {
            for (int c = 0; c < GRID; c++) {
                int32_t b  = bz[r][c];
                int32_t bu = r ? bz[r-1][c] : 0;
                int32_t bl = c ? bz[r][c-1] : 0;
                int16_t eyn = q313(ca_ey[r][c] * ey[r][c] + CB * (b - bu));
                int16_t exn = q313(ca_ex[r][c] * ex[r][c] - CB * (b - bl));
                ey[r][c] = (r == 0 || r == GRID-1) ? 0 : eyn;
                ex[r][c] = (c == 0 || c == GRID-1) ? 0 : exn;
            }
        }

        int32_t s = (int32_t)ey[GRID/2][GRID/2]
                  + (int32_t)(2048.0 * sin(iter_count * 450.0 / 8192.0));
        ey[GRID/2][GRID/2] = (int16_t)s;
        iter_count++;

        #pragma omp parallel for num_threads(nthreads) schedule(static)
        for (int r = 0; r < GRID; r++) {
            for (int c = 0; c < GRID; c++) {
                int32_t eyd = (r < GRID-1) ? ey[r+1][c] : ey[r][c];
                int32_t exr = (c < GRID-1) ? ex[r][c+1] : 0;
                bz[r][c] = q313(ca_bz[r][c] * bz[r][c]
                          + CB * ((eyd - ey[r][c]) - (exr - ex[r][c])));
            }
        }
    }
}

void build_magnitude(int mag_mode, int nthreads) {
    #pragma omp parallel for num_threads(nthreads) schedule(static)
    for (int r = 0; r < GRID; r++) {
        for (int c = 0; c < GRID; c++) {
            uint16_t ax = abs16(ex[r][c]);
            uint16_t ay = abs16(ey[r][c]);
            uint16_t hi = ax >= ay ? ax : ay;
            uint16_t lo = ax >= ay ? ay : ax;
            uint32_t e_approx = (uint32_t)hi + ((uint32_t)lo >> 1);
            if (mag_mode) {
                uint32_t b = abs16(bz[r][c]);
                mag[r][c] = sat16_u32((e_approx * b) >> 13);
            } else {
                mag[r][c] = sat16_u32(e_approx);
            }
        }
    }
}

static inline float sample_mag(float x, float y) {
    if (x < 0.0f || y < 0.0f || x > (float)(GRID - 1) || y > (float)(GRID - 1)) return 0.0f;
    int x0 = (int)x;
    int y0 = (int)y;
    int x1 = x0 < GRID - 1 ? x0 + 1 : x0;
    int y1 = y0 < GRID - 1 ? y0 + 1 : y0;
    float fx = x - (float)x0;
    float fy = y - (float)y0;
    float a = (float)mag[y0][x0];
    float b = (float)mag[y0][x1];
    float c = (float)mag[y1][x0];
    float d = (float)mag[y1][x1];
    return (a * (1.0f - fx) + b * fx) * (1.0f - fy) +
           (c * (1.0f - fx) + d * fx) * fy;
}

static inline float surface_height(float x, float y) {
    float m = sample_mag(x, y) / 4096.0f;
    if (m > 2.0f) m = 2.0f;
    return 7.0f + 20.0f * m;
}

static inline void norm3(float *x, float *y, float *z) {
    float l = sqrtf((*x)*(*x) + (*y)*(*y) + (*z)*(*z));
    if (l > 1.0e-12f) {
        *x /= l;
        *y /= l;
        *z /= l;
    }
}

int render(int nthreads) {
    const float cam_x = -28.0f, cam_y = -28.0f, cam_z = 72.0f;
    const float tgt_x = 64.0f,  tgt_y = 64.0f,  tgt_z = 8.0f;
    float fwd_x = tgt_x - cam_x, fwd_y = tgt_y - cam_y, fwd_z = tgt_z - cam_z;
    norm3(&fwd_x, &fwd_y, &fwd_z);

    float right_x = fwd_y, right_y = -fwd_x, right_z = 0.0f;
    norm3(&right_x, &right_y, &right_z);
    float up_x = right_y * fwd_z - right_z * fwd_y;
    float up_y = right_z * fwd_x - right_x * fwd_z;
    float up_z = right_x * fwd_y - right_y * fwd_x;
    norm3(&up_x, &up_y, &up_z);

    const float aspect = (float)WIDTH / (float)HEIGHT;
    const float tan_half_fov = 0.520567f;
    const float t_min = 1.0f;
    const float t_step = 190.0f / (float)RENDER_STEPS;
    const float light_x = -0.35f, light_y = -0.45f, light_z = 0.82f;
    int total_hits = 0;

    #pragma omp parallel for num_threads(nthreads) schedule(static) reduction(+:total_hits)
    for (int py = 0; py < HEIGHT; py++) {
        for (int px = 0; px < WIDTH; px++) {
            float sx = ((2.0f * ((float)px + 0.5f) / (float)WIDTH) - 1.0f) * aspect * tan_half_fov;
            float sy = (1.0f - (2.0f * ((float)py + 0.5f) / (float)HEIGHT)) * tan_half_fov;
            float dx = fwd_x + right_x * sx + up_x * sy;
            float dy = fwd_y + right_y * sx + up_y * sy;
            float dz = fwd_z + right_z * sx + up_z * sy;
            norm3(&dx, &dy, &dz);

            int hit = 0;
            float hx = 0.0f, hy = 0.0f, hz = 0.0f;
            for (int s = 0; s < RENDER_STEPS; s++) {
                float t = t_min + (float)s * t_step;
                float wx = cam_x + dx * t;
                float wy = cam_y + dy * t;
                float wz = cam_z + dz * t;
                if (wx >= 0.0f && wy >= 0.0f && wx <= (float)(GRID-1) && wy <= (float)(GRID-1)) {
                    float sh = surface_height(wx, wy);
                    if (wz <= sh) {
                        hit = 1;
                        hx = wx;
                        hy = wy;
                        hz = sh;
                        break;
                    }
                }
            }

            if (!hit) {
                float k = (float)py / (float)(HEIGHT - 1);
                frame[py][px][0] = sat8_float(122.0f - 25.0f * k);
                frame[py][px][1] = sat8_float(198.0f - 52.0f * k);
                frame[py][px][2] = sat8_float(216.0f - 58.0f * k);
                continue;
            }

            total_hits++;
            float h_l = surface_height(hx - 1.0f, hy);
            float h_r = surface_height(hx + 1.0f, hy);
            float h_d = surface_height(hx, hy - 1.0f);
            float h_u = surface_height(hx, hy + 1.0f);
            float nx = h_l - h_r;
            float ny = h_d - h_u;
            float nz = 2.0f;
            norm3(&nx, &ny, &nz);
            float diffuse = nx * light_x + ny * light_y + nz * light_z;
            if (diffuse < 0.05f) diffuse = 0.05f;
            if (diffuse > 1.0f) diffuse = 1.0f;
            float field = sample_mag(hx, hy) / 4096.0f;
            if (field > 1.5f) field = 1.5f;

            float fog = hz / 32.0f;
            if (fog > 1.0f) fog = 1.0f;
            frame[py][px][0] = sat8_float((35.0f + 90.0f * field + 60.0f * diffuse) * (0.85f + 0.15f * fog));
            frame[py][px][1] = sat8_float((70.0f + 70.0f * diffuse + 35.0f * field) * (0.90f + 0.10f * fog));
            frame[py][px][2] = sat8_float((92.0f + 42.0f * diffuse + 18.0f * field) * (0.95f + 0.05f * fog));
        }
    }

    return total_hits;
}

int run_pipeline(int frames, int solver_steps_per_frame, int mag_mode, int nthreads) {
    int hits = 0;
    for (int f = 0; f < frames; f++) {
        run_fdtd(solver_steps_per_frame, nthreads);
        build_magnitude(mag_mode, nthreads);
        hits += render(nthreads);
    }
    return hits;
}

uint32_t frame_checksum(void) {
    uint32_t h = 2166136261u;
    const uint8_t *p = &frame[0][0][0];
    for (int i = 0; i < WIDTH * HEIGHT * 3; i++) {
        h ^= (uint32_t)p[i];
        h *= 16777619u;
    }
    return h;
}

uint8_t *frame_data(void) { return &frame[0][0][0]; }
'''

with open('fdtd_render_pipeline_bench.c', 'w') as f:
    f.write(c_src)

print('C source written: fdtd_render_pipeline_bench.c')

In [ ]:
import platform
import subprocess

arm = platform.machine() in ('armv7l', 'armv6l')

if arm:
    common       = ['-fopenmp']
    scalar_flags = ['-O2', '-mcpu=cortex-a9', '-mfpu=vfpv3', '-fno-tree-vectorize']
    neon_flags   = ['-O3', '-mcpu=cortex-a9', '-mfpu=neon', '-ftree-vectorize', '-ffast-math']
else:
    common       = []
    scalar_flags = ['-O2']
    neon_flags   = ['-O3', '-ffast-math']

for name, flags in [('fdtd_render_scalar.so', scalar_flags), ('fdtd_render_neon.so', neon_flags)]:
    cmd = ['cc'] + flags + common + ['-shared', '-fPIC', '-o', name, 'fdtd_render_pipeline_bench.c', '-lm']
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(name, 'FAILED')
        print(r.stderr)
    else:
        print(name, 'ok')

if not arm:
    print('note: this is not running on the PYNQ ARM CPU; use local results only for validation')

In [ ]:
import ctypes
import time

GRID = 128
WIDTH, HEIGHT = 640, 480
RENDER_STEPS = 48
SOLVER_ITERS = 160
RENDER_FRAMES = 2
PIPELINE_FRAMES = 3
SOLVER_STEPS_PER_FRAME = 1
MAG_MODE = 0  # 0 = |E| approximation, 1 = |S| approximation

ramp = (ctypes.c_int32 * 6)(8192, 8174, 8045, 7695, 7014, 5892)

def load(path):
    lib = ctypes.CDLL('./' + path)
    lib.init.argtypes = [ctypes.POINTER(ctypes.c_int32)]
    lib.run_fdtd.argtypes = [ctypes.c_int, ctypes.c_int]
    lib.build_magnitude.argtypes = [ctypes.c_int, ctypes.c_int]
    lib.render.argtypes = [ctypes.c_int]
    lib.render.restype = ctypes.c_int
    lib.run_pipeline.argtypes = [ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int]
    lib.run_pipeline.restype = ctypes.c_int
    lib.frame_checksum.restype = ctypes.c_uint32
    lib.frame_data.restype = ctypes.POINTER(ctypes.c_uint8)
    lib.init(ramp)
    return lib

def timed(fn, *args):
    t0 = time.perf_counter()
    result = fn(*args)
    return time.perf_counter() - t0, result

def bench_solver(lib, threads):
    lib.reset()
    lib.run_fdtd(8, threads)
    lib.reset()
    dt, _ = timed(lib.run_fdtd, SOLVER_ITERS, threads)
    field_updates = 3 * GRID * GRID * SOLVER_ITERS
    return field_updates / dt, dt

def bench_render(lib, threads):
    lib.reset()
    lib.run_fdtd(24, threads)
    lib.build_magnitude(MAG_MODE, threads)
    lib.render(threads)
    t0 = time.perf_counter()
    hits = 0
    for _ in range(RENDER_FRAMES):
        hits += lib.render(threads)
    dt = time.perf_counter() - t0
    return RENDER_FRAMES / dt, hits / RENDER_FRAMES, dt

def bench_pipeline(lib, threads):
    lib.reset()
    lib.run_pipeline(1, SOLVER_STEPS_PER_FRAME, MAG_MODE, threads)
    lib.reset()
    dt, hits = timed(lib.run_pipeline, PIPELINE_FRAMES, SOLVER_STEPS_PER_FRAME, MAG_MODE, threads)
    return PIPELINE_FRAMES / dt, hits / PIPELINE_FRAMES, dt, lib.frame_checksum()

configs = [
    ('1 core scalar', 'fdtd_render_scalar.so', 1),
    ('2 core scalar', 'fdtd_render_scalar.so', 2),
    ('2 core + NEON', 'fdtd_render_neon.so', 2),
]

rows = []
for label, so, threads in configs:
    lib = load(so)
    solver_rate, solver_dt = bench_solver(lib, threads)
    render_fps, render_hits, render_dt = bench_render(lib, threads)
    pipeline_fps, pipeline_hits, pipeline_dt, checksum = bench_pipeline(lib, threads)
    rows.append({
        'config': label,
        'solver_rate': solver_rate,
        'solver_dt': solver_dt,
        'render_fps': render_fps,
        'render_hits': render_hits,
        'pipeline_fps': pipeline_fps,
        'pipeline_hits': pipeline_hits,
        'pipeline_dt': pipeline_dt,
        'checksum': checksum,
    })

print(f"{'config':<16}{'FDTD updates':>16}{'render fps':>14}{'pipeline fps':>16}{'hits/frame':>13}{'checksum':>12}")
for r in rows:
    print(f"{r['config']:<16}{r['solver_rate']/1e6:>12.1f} M/s{r['render_fps']:>14.3f}{r['pipeline_fps']:>16.3f}{r['pipeline_hits']:>13.0f}{r['checksum']:>12x}")

In [ ]:
# Adjust these if your implemented FPGA numbers change.
FPGA_SOLVER_FIELD_UPDATES_PER_S = 2.4e9
FPGA_RENDER_FPS = 60.0

fpga_solver_limited_fps = FPGA_SOLVER_FIELD_UPDATES_PER_S / (3 * GRID * GRID * SOLVER_STEPS_PER_FRAME)
fpga_pipeline_fps = min(FPGA_RENDER_FPS, fpga_solver_limited_fps)
best_cpu = max(rows, key=lambda r: r['pipeline_fps'])

active_pixels_per_s = WIDTH * HEIGHT * fpga_pipeline_fps
march_samples_per_s = active_pixels_per_s * RENDER_STEPS

print(f"FPGA solver-limited rate : {fpga_solver_limited_fps:8.1f} pipeline frames/s")
print(f"FPGA HDMI-limited rate   : {FPGA_RENDER_FPS:8.1f} pipeline frames/s")
print(f"FPGA pipeline rate used  : {fpga_pipeline_fps:8.1f} pipeline frames/s")
print(f"FPGA active pixels       : {active_pixels_per_s/1e6:8.2f} Mpix/s")
print(f"FPGA march samples       : {march_samples_per_s/1e6:8.1f} Msample/s")
print()
print(f"Best CPU pipeline        : {best_cpu['config']} = {best_cpu['pipeline_fps']:.3f} frames/s")
print(f"FPGA vs best CPU         : {fpga_pipeline_fps / best_cpu['pipeline_fps']:.1f}x faster")

for r in rows:
    print(f"FPGA vs {r['config']:<13}: {fpga_pipeline_fps / r['pipeline_fps']:.1f}x")

In [ ]:
# Optional: preview the last generated CPU frame.
try:
    import numpy as np
    import matplotlib.pyplot as plt

    preview_lib = load('fdtd_render_neon.so')
    preview_lib.reset()
    preview_lib.run_pipeline(1, SOLVER_STEPS_PER_FRAME, MAG_MODE, 2)
    ptr = preview_lib.frame_data()
    arr = np.ctypeslib.as_array(ptr, shape=(HEIGHT * WIDTH * 3,)).reshape((HEIGHT, WIDTH, 3))
    plt.figure(figsize=(8, 6))
    plt.imshow(arr)
    plt.axis('off')
    plt.title(f"CPU renderer preview, checksum {preview_lib.frame_checksum():08x}")
except Exception as e:
    print('preview skipped:', e)